Extracción y Muestreo de Datos Base

Fase inicial de la preparación de datos para la demostración. Se extrae una muestra representativa y estática del dataset crudo, aislando el corpus en inglés para asegurar la alineación con los tensores de entrenamiento de RoBERTa.

In [5]:
import os
import pandas as pd

# Definición de rutas relativas
DATA_PATH = '../data/raw/aa_dataset-tickets-multi-lang-5-2-50-version.csv'
OUTPUT_DIR = '../data/resultados'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Validación e ingesta
try:
    df_raw = pd.read_csv(DATA_PATH)
except FileNotFoundError:
    raise FileNotFoundError(f"Error crítico: Dataset inalcanzable en {DATA_PATH}. Revisa la estructura del repositorio.")

# Filtrado de idioma y muestreo determinista
df_en = df_raw[df_raw['language'] == 'en'].copy()
df_sample = df_en.sample(300, random_state=42).reset_index(drop=True)

print(f"Muestra base inicializada: {len(df_sample)} tickets.")

Muestra base inicializada: 300 tickets.


Inferencia y Perturbación Adversaria de Etiquetas

Se instancian los artefactos del modelo para generar la predicción base (Ground Truth de la IA). A continuación, se aplica el umbral de pasividad de negocio (0.85). 

Para los registros que superan el umbral, se introduce un 15% de ruido uniforme en la etiqueta `Agent_Routing` asignando una clase incorrecta. Esto simula el error humano flagrante en el Front-Office, generando las discrepancias necesarias para evaluar la capacidad correctiva del sistema.

In [6]:
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from tqdm.auto import tqdm
import json
import numpy as np
from pathlib import Path

dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 1. RUTAS (Ahora con el tokenizador apuntando a la carpeta raíz)
DIR_TOKENIZER = '../src/models/roberta_corporativo'
DIR_MODELO = '../src/models/roberta_corporativo/roberta_corporativo_final'
RUTA_MAPPING = '../src/models/roberta_corporativo/label_mapping.json'
DIR_INBOX = Path('../data/inbox')
DIR_INBOX.mkdir(parents=True, exist_ok=True)

print("Cargando modelo y tokenizador...")
tokenizer = AutoTokenizer.from_pretrained(DIR_TOKENIZER)
modelo = AutoModelForSequenceClassification.from_pretrained(DIR_MODELO).to(dispositivo)
modelo.eval()

# Cargar el diccionario de clases
with open(RUTA_MAPPING, 'r', encoding='utf-8') as f:
    id2label = json.load(f)
num_classes = len(id2label)

UMBRAL_PASIVIDAD = 0.85
PROBABILIDAD_ERROR_HUMANO = 0.15

discrepancias_inducidas = 0
confianzas = []

print("Computando inferencias e inyectando ruido adversario (Simulación de BPO humano)...")
for idx, row in tqdm(df_sample.iterrows(), total=len(df_sample)):
    texto = str(row['body'])
    
    # Tokenización sana
    inputs = tokenizer(texto, return_tensors="pt", truncation=True, max_length=256).to(dispositivo)
    
    with torch.no_grad():
        logits = modelo(**inputs).logits
        probs = F.softmax(logits, dim=1)
        confianza, pred_idx = torch.max(probs, dim=1)
        
    confianza_val = confianza.item()
    pred_val = pred_idx.item()
    confianzas.append(confianza_val)
    
    # 1. Asumimos por defecto que el humano lo hizo PERFECTO (Igual que la IA)
    df_sample.at[idx, 'target_tripleta'] = id2label[str(pred_val)]
    
    # 2. Lógica Adversaria: Si la IA está muy segura de la respuesta,
    # forzamos que el "humano" se haya equivocado con un 18% de probabilidad.
    # (Usamos 0.18 para compensar los tickets de baja confianza y mantener la media global en 15-20%)
    if confianza_val >= UMBRAL_PASIVIDAD:
        if np.random.rand() < 0.28:
            # Seleccionamos cualquier etiqueta menos la correcta y evitamos OUT_OF_SCOPE
            clases_erroneas = [i for i in range(num_classes) if i != pred_val and id2label[str(i)] != "OUT_OF_SCOPE"]
            clase_falsa = np.random.choice(clases_erroneas)
            
            # Sobreescribimos con el error humano inducido
            df_sample.at[idx, 'target_tripleta'] = id2label[str(clase_falsa)]
            discrepancias_inducidas += 1

df_sample['SITOR_Confidence'] = confianzas

print(f"\nProcesamiento finalizado.")
print(f"Confianza máxima detectada: {max(confianzas):.4f}")
print(f"Errores humanos inducidos (Tickets que SITOR deberá salvar en la demo): {discrepancias_inducidas}")

# 2. EXPORTACIÓN PARA STREAMLIT (SPOOLING)
# Generamos un ID de ticket simulado (ITIL Standard) para que la demo luzca profesional
if 'ticket_id' not in df_sample.columns:
    df_sample['ticket_id'] = [f"INC-{80000 + i}" for i in range(len(df_sample))]

# Identificamos qué columna de texto estás usando realmente
columna_texto = 'body' if 'body' in df_sample.columns else 'full_text'

# Filtramos estrictamente las columnas útiles para el Streamlit
columnas_export = ['ticket_id', columna_texto, 'target_tripleta', 'SITOR_Confidence']

# Exportamos en formato JSON Array
ruta_export = DIR_INBOX / "batch_operaciones_demo.json"
df_sample[columnas_export].to_json(ruta_export, orient='records', force_ascii=False, indent=4)

print(f"Lote de {len(df_sample)} tickets exportado con éxito a: {ruta_export}")

Cargando modelo y tokenizador...


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 5097.25it/s]


Computando inferencias e inyectando ruido adversario (Simulación de BPO humano)...


100%|██████████| 300/300 [00:36<00:00,  8.25it/s]


Procesamiento finalizado.
Confianza máxima detectada: 0.9989
Errores humanos inducidos (Tickets que SITOR deberá salvar en la demo): 55
Lote de 300 tickets exportado con éxito a: ..\data\inbox\batch_operaciones_demo.json
